# Day 2 Homework — Build Your Own Saudi Agent
### Agentic AI Bootcamp · Saudi Digital Academy × atomcamp

Combine today's three skills into ONE small working agent:
1. an **engineered prompt + context**,
2. a **LangChain agent with at least one tool + structured output**,
3. a **Gradio interface**.

Fill in every `TODO`. Keep your scenario small — one job, done well.

## 0 · Setup

In [144]:
!pip install -q langchain langchain-classic langchain-openai gradio pydantic

In [145]:

import os, getpass, json
os.environ['OPENAI_API_KEY'] = getpass.getpass('Groq API key: ')  # free key: console.groq.com/keys
os.environ['OPENAI_BASE_URL'] = 'https://api.groq.com/openai/v1'  # Groq (free)
os.environ['OPENAI_API_BASE'] = 'https://api.groq.com/openai/v1'

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='openai/gpt-oss-120b', temperature=0)

Groq API key: ··········


## 1 · Your agent's goal (one sentence)
**TODO:** write what your agent does, and which Saudi sector it serves.

In [146]:
# Example: 'An academic advisor that recommends 3 electives for a Saudi CS student.'
AGENT_GOAL = 'A hospital triage guide for Saudi healthcare: directs a patient to the right department (no diagnosis).'

## 2 · Engineered prompt + context (the 6 elements)
Role · Task · Context · Reasoning · Output format · Constraints.

In [147]:
SYSTEM_PROMPT = '''Role: You are a hospital triage guide for patients in Saudi Arabia. You direct patients to the right department. You are NOT a doctor and you never diagnose.
Audience: Patients with no medical background. Use simple words.
Context: The user message contains the patient's complaint, the patient's location (it may be "not provided"), the facility type the patient wants ("hospital" or "clinic") and the language to answer in.
Task and tools:
1. ALWAYS call triage_lookup first, with the patient's complaint.
2. If a location is provided, call find_nearest_facilities once, with that location and with facility_type set to the facility type the patient wants. If the location is "not provided", do not call it.
3. Then write the final answer.
Reasoning: Think step-by-step before answering, but output only the final JSON.
Constraints: Do not diagnose or name diseases. Do not suggest medicines. Do not add details the patient did not mention (such as severity or duration). Do not state phone numbers, opening hours, crowd levels or waiting times: the app shows verified information separately. Do not list hospitals or clinics in your answer. If triage_lookup says urgent is true, or the complaint mentions severe symptoms (chest pain, difficulty breathing, heavy bleeding, loss of consciousness), set urgent to true. Write every JSON value in the requested answer language only, without mixing languages. JSON keys stay in English.
Output format: Return ONLY JSON with exactly these keys: "department" (string), "urgent" (true or false), "reason" (string, one short sentence), "next_steps" (list of 2 or 3 short strings). No other keys and no markdown.'''

## 3 · Add ONE Python tool
A tool is just a Python function the agent can use. Keep it simple.

In [148]:
import json
import math
import time
import threading
import requests
from urllib.parse import quote
from langchain_core.tools import tool

# ---------- Rules (sample data for a student project, not official medical guidance) ----------
KEYWORDS = [
    ('chest pain', 'Emergency', True),
    ('ألم في الصدر', 'Emergency', True),
    ('shortness of breath', 'Emergency', True),
    ('difficulty breathing', 'Emergency', True),
    ('ضيق في التنفس', 'Emergency', True),
    ('heavy bleeding', 'Emergency', True),
    ('نزيف شديد', 'Emergency', True),
    ('unconscious', 'Emergency', True),
    ('فقدان الوعي', 'Emergency', True),
    ('rash', 'Dermatology', False),
    ('طفح', 'Dermatology', False),
    ('حساسية جلدية', 'Dermatology', False),
    ('tooth', 'Dental', False),
    ('ألم في الأسنان', 'Dental', False),
    ('eye', 'Ophthalmology', False),
    ('ألم في العين', 'Ophthalmology', False),
    ('knee', 'Orthopedics', False),
    ('back pain', 'Orthopedics', False),
    ('ألم في الركبة', 'Orthopedics', False),
    ('ألم في الظهر', 'Orthopedics', False),
    ('headache', 'General Practice', False),
    ('صداع', 'General Practice', False),
    ('fever', 'General Practice', False),
    ('حرارة', 'General Practice', False),
]

DEPT_NAMES = {
    'Emergency': {'English': 'Emergency', 'Arabic': 'الطوارئ'},
    'Dermatology': {'English': 'Dermatology', 'Arabic': 'الجلدية'},
    'Dental': {'English': 'Dental', 'Arabic': 'الأسنان'},
    'Ophthalmology': {'English': 'Ophthalmology', 'Arabic': 'العيون'},
    'Orthopedics': {'English': 'Orthopedics', 'Arabic': 'العظام'},
    'General Practice': {'English': 'General Practice', 'Arabic': 'الطب العام'},
}

def find_department(text):
    '''Scans the patient message for keywords. Returns the department and whether it is urgent.'''
    text = text.lower()
    matches = [(dept, urgent) for kw, dept, urgent in KEYWORDS if kw in text]
    if not matches:
        return {'department': 'General Practice', 'urgent': False, 'matched': False}
    urgent = any(u for _, u in matches)
    dept = 'Emergency' if urgent else matches[0][0]
    return {'department': dept, 'urgent': urgent, 'matched': True}

# ---------- Tool 1: triage rules ----------
@tool
def triage_lookup(complaint: str) -> str:
    """Suggest the hospital department and urgency for a patient's complaint using keyword rules. Returns JSON with department, urgent and matched."""
    return json.dumps(find_department(complaint))

# ---------- Tool 2: nearest real health facilities (OpenStreetMap data) ----------
HEADERS = {'User-Agent': 'hospital-triage-student-project/1.0 (educational project)'}
NOMINATIM_URL = 'https://nominatim.openstreetmap.org/search'
OPEN_METEO_URL = 'https://geocoding-api.open-meteo.com/v1/search'          # backup geocoder
OVERPASS_URLS = ['https://overpass-api.de/api/interpreter',                 # main public server (often overloaded)
                 'https://maps.mail.ru/osm/tools/overpass/api/interpreter', # VK Maps public server
                 'https://overpass.private.coffee/api/interpreter']         # Private.coffee public server
FALLBACK_PAUSE = 1.1          # Nominatim allows max 1 request per second
STAGGER_S = 3                 # next Overpass server starts if the previous one has not answered within 3 s
OVERPASS_WAIT_S = 22          # give up on one Overpass query after this many seconds in total
FAIL_TTL = 20                 # a failed lookup is remembered for 20 s (so a retry does not wait again)

# How each facility type is tagged in OpenStreetMap
FACILITY_FILTERS = {
    'hospital': ['["amenity"="hospital"]', '["healthcare"="hospital"]'],
    'clinic': ['["amenity"="clinic"]', '["healthcare"="clinic"]', '["healthcare"="centre"]'],
}
RADII = {'hospital': (15000, 50000), 'clinic': (5000, 15000, 40000)}     # small first = lighter query
NOMINATIM_TYPES = {'hospital': {'hospital'}, 'clinic': {'clinic', 'centre', 'doctors'}}
_CACHE = {}                   # (place, type) -> (time, ok, result JSON): repeated searches do not hit the servers again
_INFLIGHT = {}                # (place, type) -> thread currently running that lookup
_LOCK = threading.Lock()

def _distance_km(lat1, lon1, lat2, lon2):
    """Straight-line distance between two points (haversine formula)."""
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

def _short(e):
    """Short readable reason for a failed request (shown to the user for debugging)."""
    resp = getattr(e, 'response', None)
    if resp is not None and getattr(resp, 'status_code', None):
        return f'HTTP {resp.status_code}'
    if isinstance(e, requests.Timeout):
        return 'timeout'
    if isinstance(e, requests.ConnectionError):
        return 'connection error'
    return f'{type(e).__name__}: {str(e)[:50]}'

def _geocode(place):
    """Place name -> (lat, lon). Tries Nominatim first, then Open-Meteo. Returns (lat, lon, errors, nomatch)."""
    errors, nomatch = [], []
    try:
        r = requests.get(NOMINATIM_URL, headers=HEADERS, timeout=(5, 10),
                         params={'q': place, 'format': 'json', 'limit': 1, 'countrycodes': 'sa'})
        r.raise_for_status()
        data = r.json()
        if data:
            return float(data[0]['lat']), float(data[0]['lon']), errors, nomatch
        nomatch.append('nominatim: no match')
    except Exception as e:
        errors.append(f'nominatim: {_short(e)}')
    try:
        r = requests.get(OPEN_METEO_URL, headers=HEADERS, timeout=(5, 10),
                         params={'name': place, 'count': 5, 'language': 'en', 'format': 'json'})
        r.raise_for_status()
        results = r.json().get('results') or []
        if results:
            pick = next((x for x in results if x.get('country_code') == 'SA'), results[0])
            return float(pick['latitude']), float(pick['longitude']), errors, nomatch
        nomatch.append('open-meteo: no match')
    except Exception as e:
        errors.append(f'open-meteo: {_short(e)}')
    return None, None, errors, nomatch

def _overpass(query, deadline=None):
    """Asks the Overpass servers in a staggered race: server 1 now, server 2 after 3 s, server 3 after 6 s
    (only if nobody has answered yet). The first good answer wins. Returns (elements or None, errors)."""
    results, errors = [], []
    done = threading.Event()
    lock = threading.Lock()

    def worker(url, delay):
        host = url.split('/')[2]
        if delay and done.wait(delay):                    # someone already answered -> do not even ask
            return
        try:
            resp = requests.post(url, data={'data': query}, headers=HEADERS, timeout=(5, 15))
            resp.raise_for_status()
            data = resp.json()
            if 'runtime error' in str(data.get('remark', '')).lower():
                raise RuntimeError(str(data['remark'])[:60])
            with lock:
                results.append(data.get('elements', []))
            done.set()
        except Exception as e:
            with lock:
                errors.append(f'{host}: {_short(e)}')

    threads = [threading.Thread(target=worker, args=(u, i * STAGGER_S), daemon=True)
               for i, u in enumerate(OVERPASS_URLS)]
    for t in threads:
        t.start()
    end = time.monotonic() + OVERPASS_WAIT_S
    while time.monotonic() < end:
        if done.is_set():
            return results[0], []
        with lock:
            if len(errors) >= len(threads):               # every server failed
                break
        time.sleep(0.05)
    if results:
        return results[0], []
    done.set()                                            # stop servers that have not started yet
    with lock:
        seen = {e.split(':')[0] for e in errors}
        pending = [u.split('/')[2] + ': timeout' for u in OVERPASS_URLS if u.split('/')[2] not in seen]
        return None, errors + pending

def _nominatim_facilities(lat, lon, ftype):
    """Last-resort source: Nominatim search inside a box around the place. Returns (elements or None, errors)."""
    errors, found = [], {}
    for half in (0.15, 0.5):                              # about 17 km, then about 55 km
        try:
            time.sleep(FALLBACK_PAUSE)
            r = requests.get(NOMINATIM_URL, headers=HEADERS, timeout=(5, 10), params={
                'q': ftype, 'format': 'jsonv2', 'limit': 40, 'bounded': 1, 'extratags': 1, 'namedetails': 1,
                'viewbox': f'{lon - half},{lat + half},{lon + half},{lat - half}'})
            r.raise_for_status()
            for item in r.json():
                if item.get('type') not in NOMINATIM_TYPES[ftype]:
                    continue
                nd, ex = item.get('namedetails') or {}, item.get('extratags') or {}
                name = item.get('name') or nd.get('name') or nd.get('name:ar') or nd.get('name:en')
                if not name:
                    continue
                tags = {'name': name, 'name:ar': nd.get('name:ar'), 'name:en': nd.get('name:en'),
                        'emergency': ex.get('emergency'), 'opening_hours': ex.get('opening_hours'),
                        'phone': ex.get('phone') or ex.get('contact:phone')}
                found[item['osm_id']] = {'lat': float(item['lat']), 'lon': float(item['lon']), 'tags': tags}
            if found:
                break
        except Exception as e:
            errors.append(f'nominatim-search: {_short(e)}')
            return None, errors
    return list(found.values()), errors

def _failure(kind, ftype, place, errors):
    return json.dumps({'error': kind, 'place': place, 'facility_type': ftype,
                       'detail': '; '.join(errors)[:300]}, ensure_ascii=False)

def _build(elements, lat, lon, place, ftype, radius_km, source):
    best = {}                                            # one entry per name (nearest wins)
    for el in elements:
        tags = el.get('tags', {})
        point = el if 'lat' in el else el.get('center')
        name = tags.get('name') or tags.get('name:ar') or tags.get('name:en')
        if not point or not name:
            continue
        dist = round(_distance_km(lat, lon, point['lat'], point['lon']), 1)
        if name in best and best[name]['distance_km'] <= dist:
            continue
        best[name] = {
            'name': name,
            'name_ar': tags.get('name:ar'),
            'name_en': tags.get('name:en'),
            'distance_km': dist,
            'emergency': tags.get('emergency'),              # 'yes' / 'no' / None (as tagged in OSM)
            'opening_hours': tags.get('opening_hours'),
            'phone': tags.get('phone') or tags.get('contact:phone'),
            'maps_url': 'https://www.google.com/maps/search/?api=1&query=' + quote(f'{name}, {place}'),
        }
    facilities = sorted(best.values(), key=lambda h: h['distance_km'])[:3]
    return json.dumps({'place': place, 'facility_type': ftype, 'search_radius_km': radius_km,
                       'source': source, 'facilities': facilities}, ensure_ascii=False)

def _lookup(place, ftype):
    """The real work: place -> coordinates -> facilities. Returns a JSON string."""
    try:
        lat, lon, errors, nomatch = _geocode(place)
        if lat is None:
            if errors:                                   # a server failed -> real error, not "unknown place"
                return _failure('lookup_failed', ftype, place, errors + nomatch)
            return json.dumps({'error': 'place_not_found', 'place': place, 'facility_type': ftype}, ensure_ascii=False)

        elements, used_radius, errs = [], 0, []
        for radius in RADII[ftype]:                      # widen the search if nothing is found
            parts = ''.join(f'nwr(around:{radius},{lat},{lon}){f};' for f in FACILITY_FILTERS[ftype])
            elements, errs = _overpass(f'[out:json][timeout:20];({parts});out center 100;')
            if elements is None:
                break
            used_radius = radius
            if elements:
                break

        if elements is not None:                         # Overpass answered
            result = _build(elements, lat, lon, place, ftype, used_radius // 1000, 'overpass')
        else:                                            # every Overpass server failed -> Nominatim fallback
            elements, nerrs = _nominatim_facilities(lat, lon, ftype)
            if elements is None:
                return _failure('lookup_failed', ftype, place, errs + nerrs)
            result = _build(elements, lat, lon, place, ftype, 55, 'nominatim')
        return result
    except Exception as e:
        return _failure('lookup_failed', ftype, place, [_short(e)])

def _cached(place, ftype):
    """Returns the cached JSON (ok results stay for the session; failures only for FAIL_TTL seconds)."""
    key = (str(place).strip().lower(), ftype)
    item = _CACHE.get(key)
    if item and (item[1] or time.time() - item[0] < FAIL_TTL):
        return item[2]
    return None

def _run_and_store(place, ftype):
    key = (str(place).strip().lower(), ftype)
    result = _lookup(place, ftype)
    try:
        ok = 'error' not in json.loads(result)
        not_found = json.loads(result).get('error') == 'place_not_found'
    except Exception:
        ok, not_found = False, False
    _CACHE[key] = (time.time(), ok or not_found, result)
    return result

def prefetch_facilities(place, facility_type='hospital'):
    """Starts the lookup in the background (called as soon as the user presses the button),
    so the result is usually ready when the agent decides to call the tool."""
    ftype = 'clinic' if str(facility_type).strip().lower().startswith('clin') else 'hospital'
    key = (str(place).strip().lower(), ftype)
    if not key[0]:
        return
    with _LOCK:
        if _cached(place, ftype) is not None or key in _INFLIGHT:
            return
        t = threading.Thread(target=lambda: (_run_and_store(place, ftype), _INFLIGHT.pop(key, None)), daemon=True)
        _INFLIGHT[key] = t
    t.start()

@tool
def find_nearest_facilities(place: str, facility_type: str = 'hospital') -> str:
    """Find the nearest real health facilities, from OpenStreetMap data, to a city or district in Saudi Arabia. place: a place name such as 'Jazan' or 'جازان'. facility_type: 'hospital' or 'clinic' (clinics and health centers). Returns JSON sorted by distance. It does NOT contain crowd levels or waiting times."""
    ftype = 'clinic' if str(facility_type).strip().lower().startswith('clin') else 'hospital'
    key = (str(place).strip().lower(), ftype)
    hit = _cached(place, ftype)
    if hit is not None:
        return hit
    running = _INFLIGHT.get(key)
    if running is not None:                              # the background lookup is already working on it
        running.join(timeout=60)
        hit = _cached(place, ftype)
        if hit is not None:
            return hit
    return _run_and_store(place, ftype)

## 4 · Structured output with Pydantic (recommended)
Force the result into a fixed shape so it's reliable.

In [149]:
from pydantic import BaseModel, Field
from typing import List

class Result(BaseModel):
    department: str = Field(description='department the patient should visit')
    urgent: bool = Field(description='true if the patient should go to Emergency now')
    reason: str = Field(description='one short sentence explaining why')
    next_steps: List[str] = Field(description='2-3 simple next steps')

from langchain_classic.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=Result)
print(parser.get_format_instructions()[:300])

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]}


## 5 · Put it together: prompt → model → structured output
A simple LCEL pipeline (modern LangChain): `prompt | llm | parser`.

In [150]:
import langchain
from langchain.agents import create_agent

print('langchain version:', langchain.__version__)

# The model decides which tools to call (triage_lookup, find_nearest_facilities) and when.
agent = create_agent(
    model=llm,
    tools=[triage_lookup, find_nearest_facilities],
    system_prompt=SYSTEM_PROMPT,
)

def _text(content):
    '''Message content can be a string or a list of blocks.'''
    if isinstance(content, str):
        return content
    return ''.join(b.get('text', '') if isinstance(b, dict) else str(b) for b in content)

def ask(request, language, location='', facility_type='hospital'):
    '''Runs the agent. Returns (structured result, rule-based info, facility data from the tool).'''
    info = find_department(request)          # safety rules run in code, independent of the model
    if location.strip():                     # start the facility search now, in the background, to save time
        prefetch_facilities(location.strip(), facility_type)
    user_msg = (f'Patient complaint: {request}\n'
                f"Patient location: {location.strip() or 'not provided'}\n"
                f'Facility type wanted: {facility_type}\n'
                f'Answer language: {language}')

    last_error = None
    for _ in range(2):                       # one retry if the model's JSON cannot be parsed
        try:
            out = agent.invoke({'messages': [{'role': 'user', 'content': user_msg}]})
            msgs = out['messages']
            result = parser.parse(_text(msgs[-1].content))
            break
        except Exception as e:
            last_error = e
    else:
        raise last_error

    # Facility facts come from the tool output, not from the model text
    calls = []
    for m in msgs:
        if getattr(m, 'type', '') == 'tool' and getattr(m, 'name', '') == 'find_nearest_facilities':
            try:
                calls.append(json.loads(_text(m.content)))
            except Exception:
                pass
    facilities = next((c for c in calls if c.get('facility_type') == facility_type), calls[-1] if calls else None)

    # Safety net: a location was given but the model skipped the tool or asked for the wrong type
    if location.strip() and (facilities is None or facilities.get('facility_type') != facility_type):
        try:
            facilities = json.loads(find_nearest_facilities.invoke(
                {'place': location.strip(), 'facility_type': facility_type}))
        except Exception:
            pass
    return result, info, facilities

# Quick test (needs internet for the facility lookup)
try:
    r, info, fac = ask('I have a rash on my arm for a week', 'English', 'Jazan', 'hospital')
    print(r)
    print(info)
    print(fac)
except Exception as e:
    print('Check the agent/tool setup:', e)

langchain version: 1.4.8
department='Dermatology' urgent=False reason='You have a skin rash that needs a specialist.' next_steps=['Go to the nearest hospital dermatology department.', 'Bring any pictures of the rash if possible.']
{'department': 'Dermatology', 'urgent': False, 'matched': True}
{'place': 'Jazan', 'facility_type': 'hospital', 'search_radius_km': 55, 'source': 'nominatim', 'facilities': [{'name': 'مستشفى بيش العام', 'name_ar': 'مستشفى بيش العام', 'name_en': 'Baish General Hospital', 'distance_km': 18.8, 'emergency': None, 'opening_hours': None, 'phone': None, 'maps_url': 'https://www.google.com/maps/search/?api=1&query=%D9%85%D8%B3%D8%AA%D8%B4%D9%81%D9%89%20%D8%A8%D9%8A%D8%B4%20%D8%A7%D9%84%D8%B9%D8%A7%D9%85%2C%20Jazan'}]}


## 6 · Wrap it in Gradio
textbox in → your agent → structured result. `share=True` gives a public link.

In [151]:
import gradio as gr
import inspect
import threading
import traceback

# ============ Texts ============
UI = {
    'العربية': {
        'lang_name': 'Arabic',
        'goal': 'اكتب ما تشعر به وسنرشدك إلى القسم المناسب وأقرب المستشفيات.',
        'input_label': 'اكتب شكواك',
        'placeholder': 'مثال: عندي طفح جلدي في يدي من أسبوع',
        'loc_label': 'موقعك (اختياري)',
        'loc_placeholder': 'مثال: جازان',
        'fac_label': 'ابحث عن',
        'fac_hospital': 'مستشفيات',
        'fac_clinic': 'مستوصفات ومراكز صحية',
        'analyzing': 'جارٍ تحليل شكواك والبحث عن أقرب المرافق... قد يستغرق ذلك بضع ثوانٍ.',
        'technical': 'السبب التقني',
        'find': 'ابحث عن القسم', 'back': 'رجوع', 'new': 'استشارة جديدة',
        'too_short': 'الرجاء كتابة وصف أوضح لشكواك (جملة قصيرة على الأقل).',
        'urgent_banner': 'حالة عاجلة: توجّه إلى الطوارئ فورًا.',
        'urgent_call': 'للإسعاف (الهلال الأحمر السعودي) اتصل على 997.',
        'department': 'القسم', 'reason': 'السبب', 'steps_title': 'الخطوات التالية',
        'hosp_title': 'أقرب المستشفيات', 'clinic_title': 'أقرب المستوصفات والمراكز الصحية', 'km': 'كم', 'hours': 'الدوام', 'phone': 'الهاتف',
        'er_yes': 'فيها قسم طوارئ', 'open_maps': 'افتح في خرائط Google',
        'none_found': 'لم أجد مرافق قريبة من هذا النوع في البيانات المتاحة. جرّب النوع الآخر.',
        'place_not_found': 'ما قدرت أحدد الموقع. جرّب اسم مدينة أو حي أوضح.',
        'hosp_unavailable': 'تعذّر جلب المرافق القريبة الآن، حاول مرة أخرى.',
        'disclaimer': 'مشروع تعليمي غير تابع لأي جهة حكومية، ولا يقدّم تشخيصًا طبيًا.',
        'error': 'حدث خطأ، حاول مرة أخرى.',
        'steps': ['اللغة', 'الشكوى', 'النتيجة'],
        'examples_label': 'أمثلة',
        'examples': ['عندي صداع من يومين', 'طفح جلدي في يدي من أسبوع', 'ألم في الصدر وضيق في التنفس'],
    },
    'English': {
        'lang_name': 'English',
        'goal': AGENT_GOAL,
        'input_label': 'Describe your complaint',
        'placeholder': 'Example: I have had a rash on my arm for a week',
        'loc_label': 'Your location (optional)',
        'loc_placeholder': 'Example: Jazan',
        'fac_label': 'Search for',
        'fac_hospital': 'Hospitals',
        'fac_clinic': 'Clinics & health centers',
        'analyzing': 'Analyzing your complaint and searching for nearby facilities... this may take a few seconds.',
        'technical': 'Technical reason',
        'find': 'Find my department', 'back': 'Back', 'new': 'New consultation',
        'too_short': 'Please describe your complaint in more detail (a short sentence at least).',
        'urgent_banner': 'Urgent: go to Emergency immediately.',
        'urgent_call': 'For an ambulance (Saudi Red Crescent) call 997.',
        'department': 'Department', 'reason': 'Reason', 'steps_title': 'Next steps',
        'hosp_title': 'Nearest hospitals', 'clinic_title': 'Nearest clinics & health centers', 'km': 'km', 'hours': 'Hours', 'phone': 'Phone',
        'er_yes': 'has an emergency department', 'open_maps': 'Open in Google Maps',
        'none_found': 'No nearby facilities of this type were found in the available data. Try the other type.',
        'place_not_found': 'I could not find that location. Try a clearer city or district name.',
        'hosp_unavailable': 'Could not fetch nearby facilities right now, please try again.',
        'disclaimer': 'Educational project, not affiliated with any government entity, and gives no medical diagnosis.',
        'error': 'Something went wrong, please try again.',
        'steps': ['Language', 'Complaint', 'Result'],
        'examples_label': 'Examples',
        'examples': ['I have had a headache for two days', 'I have a rash on my arm for a week', 'Chest pain and shortness of breath'],
    },
}
DEFAULT_STEPS = ['اللغة · Language', 'الشكوى · Complaint', 'النتيجة · Result']

# ============ Design: colors + animations ============
CSS = '''
:root { --brand:#0f766e; --brand2:#14b8a6; --accent:#10b981; }
.gradio-container { max-width: 880px !important; margin: auto !important; }

@keyframes slideUp { from {opacity:0; transform:translateY(26px);} to {opacity:1; transform:translateY(0);} }
@keyframes fadeIn { from {opacity:0;} to {opacity:1;} }
@keyframes gradientShift { 0%{background-position:0% 50%;} 50%{background-position:100% 50%;} 100%{background-position:0% 50%;} }
@keyframes ecg { 0%{stroke-dashoffset:100; opacity:1;} 70%{stroke-dashoffset:0; opacity:1;} 100%{stroke-dashoffset:0; opacity:.15;} }
@keyframes pulseAlert { 0%,100%{box-shadow:0 0 0 0 rgba(220,38,38,.55);} 50%{box-shadow:0 0 0 14px rgba(220,38,38,0);} }
@keyframes shake { 0%,100%{transform:translateX(0);} 25%{transform:translateX(-6px);} 75%{transform:translateX(6px);} }
@keyframes itemIn { from {opacity:0; transform:translateX(-14px);} to {opacity:1; transform:none;} }

/* Hero header */
.hero { border-radius:20px; padding:22px 26px; background:linear-gradient(120deg,#0f766e,#14b8a6,#10b981,#0f766e);
        background-size:300% 300%; animation:gradientShift 10s ease infinite, fadeIn .8s ease both; }
.hero h1, .hero p { color:#fff !important; margin:0; }
.hero h1 { font-size:1.5rem; }
.hero p { margin-top:4px; opacity:.92; }
.ecg { width:100%; height:34px; margin-top:12px; display:block; }
.ecg path { stroke:#fff; stroke-width:2.4; fill:none; stroke-linecap:round; stroke-linejoin:round;
            vector-effect:non-scaling-stroke; stroke-dasharray:100; animation:ecg 2.8s linear infinite; }

/* Stepper */
.stepper { display:flex; align-items:center; justify-content:center; gap:10px; margin:16px 0 6px; }
.step { display:flex; align-items:center; gap:8px; opacity:.5; font-weight:600; transition:all .4s ease; }
.step .dot { width:30px; height:30px; border-radius:50%; display:grid; place-items:center;
             background:#cbd5e1; color:#fff; transition:all .4s ease; }
.step.active { opacity:1; }
.step.active .dot { background:var(--brand); transform:scale(1.15); box-shadow:0 0 0 6px rgba(20,184,166,.22); }
.step.done { opacity:.9; }
.step.done .dot { background:var(--accent); }
.bar { width:44px; height:3px; background:#cbd5e1; border-radius:3px; transition:background .4s ease; }
.bar.done { background:var(--accent); }

/* Screens: animate every time one becomes visible */
.screen { animation:slideUp .55s cubic-bezier(.2,.8,.2,1) both; }

/* Buttons */
.lang-btn, .lang-btn button { font-size:1.3rem !important; padding:26px 10px !important; border-radius:18px !important;
                              transition:transform .25s ease, box-shadow .25s ease !important; }
.lang-btn:hover, .lang-btn button:hover { transform:translateY(-6px) scale(1.02); box-shadow:0 14px 30px rgba(15,118,110,.3); }
button.primary { transition:transform .2s ease, box-shadow .2s ease !important; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 8px 20px rgba(15,118,110,.35); }
button.primary:active { transform:scale(.97); }

/* Textbox */
textarea, input[type=text] { border-radius:14px !important; transition:box-shadow .25s ease, border-color .25s ease; unicode-bidi:plaintext; text-align:start; }
textarea:focus, input[type=text]:focus { box-shadow:0 0 0 4px rgba(20,184,166,.25) !important; border-color:var(--brand2) !important; }

/* Result card */
.result { background:#f0fdfa; border:1px solid #99f6e4; border-radius:18px; padding:18px 22px;
          box-shadow:0 10px 30px rgba(15,118,110,.12); }
.result, .result p, .result li, .result strong, .result em { color:#0b3b3a !important; }
.result h3 { color:#0f766e !important; }
.result a { color:#0f766e !important; font-weight:600; }
.result li { animation:itemIn .5s ease both; }
.result li:nth-child(1) { animation-delay:.25s; }
.result li:nth-child(2) { animation-delay:.45s; }
.result li:nth-child(3) { animation-delay:.65s; }
.result li:nth-child(4) { animation-delay:.85s; }
.result blockquote { border:0 !important; border-radius:14px; padding:12px 16px; background:#fee2e2;
                     animation:shake .5s ease, pulseAlert 1.6s ease-in-out .5s infinite; }
.result blockquote, .result blockquote p, .result blockquote strong { color:#991b1b !important; }

/* Footer: contact links */
.footer { margin-top:28px; padding:20px; border-radius:18px; text-align:center;
          border:1px solid rgba(20,184,166,.35); animation:fadeIn 1s ease both; }
.footer .name { font-weight:700; font-size:1.05rem; }
.footer .links { display:flex; gap:12px; justify-content:center; flex-wrap:wrap; margin:12px 0; }
.footer .social { display:inline-flex; align-items:center; gap:8px; padding:10px 20px; border-radius:999px;
                  text-decoration:none !important; font-weight:600; color:#fff !important;
                  background:linear-gradient(120deg,#0f766e,#14b8a6);
                  transition:transform .25s ease, box-shadow .25s ease; }
.footer .social:hover { transform:translateY(-3px); box-shadow:0 10px 22px rgba(15,118,110,.35); }
.footer .rights { font-size:.85rem; opacity:.7; }

/* No Gradio progress timer anywhere */
.progress-text, .meta-text-center, .meta-text { display:none !important; }

/* Loading card (shown while the agent works) */
@keyframes bounce { 0%,80%,100%{transform:translateY(0); opacity:.5;} 40%{transform:translateY(-8px); opacity:1;} }
.loading { display:flex; align-items:center; gap:14px; padding:28px 8px; font-weight:600; }
.loading .dots { display:inline-flex; gap:6px; }
.loading .dots i { width:10px; height:10px; border-radius:50%; background:var(--brand2); display:inline-block;
                   animation:bounce 1.2s ease-in-out infinite; }
.loading .dots i:nth-child(2) { animation-delay:.15s; }
.loading .dots i:nth-child(3) { animation-delay:.3s; }

@media (prefers-reduced-motion: reduce) { * { animation:none !important; transition:none !important; } }
'''

HERO = '''
<div class="hero">
  <h1>🏥 مرشد المستشفى الذكي | Smart Hospital Guide</h1>
  <p>مساعد توجيه المرضى · Patient triage assistant</p>
  <svg class="ecg" viewBox="0 0 400 40" preserveAspectRatio="none">
    <path pathLength="100" d="M0 20 H90 L105 20 L115 4 L127 36 L139 20 H220 L235 20 L245 2 L258 38 L270 20 H400"/>
  </svg>
</div>
'''

FOOTER = '''
<div class="footer">
  <div class="name">Eng: Abdullah Abdali</div>
  <div>تواصل معي · Connect with me</div>
  <div class="links">
    <a class="social" href="https://github.com/Eng-cloud" target="_blank" rel="noopener noreferrer">💻 GitHub</a>
    <a class="social" href="https://www.linkedin.com/in/abdullah-abdali-342126346" target="_blank" rel="noopener noreferrer">🔗 LinkedIn</a>
  </div>
  <div class="rights">© 2026 Abdullah Abdali · جميع الحقوق محفوظة · All rights reserved</div>
</div>
'''

# ============ Helpers ============
def stepper(active, labels, rtl=False):
    html = '<div class="stepper"' + (' dir="rtl"' if rtl else '') + '>'
    for i, label in enumerate(labels, start=1):
        cls = 'active' if i == active else ('done' if i < active else '')
        mark = '✓' if i < active else str(i)
        html += f'<div class="step {cls}"><span class="dot">{mark}</span><span>{label}</span></div>'
        if i < len(labels):
            html += f'<div class="bar {"done" if i < active else ""}"></div>'
    return html + '</div>'

def wrap(text, name):
    # Arabic text is shown right-to-left
    return f'<div dir="rtl">\n\n{text}\n\n</div>' if name == 'Arabic' else text

def _clean(s):
    # keep hospital names from breaking the Markdown links
    return str(s).replace('[', '(').replace(']', ')').replace('\n', ' ')

def facilities_md(data, t, name):
    '''Builds the facility list from the tool output (facts come from the tool, not from the model).'''
    if not data:
        return ''
    title = f"**{t['clinic_title'] if data.get('facility_type') == 'clinic' else t['hosp_title']}:**"
    err = data.get('error')
    if err == 'place_not_found':
        return f"{title}\n\n{t['place_not_found']}\n\n"
    if err:
        detail = f" *({t['technical']}: {_clean(data.get('detail', err))})*"
        return f"{title}\n\n{t['hosp_unavailable']}{detail}\n\n"
    items = data.get('facilities', [])
    if not items:
        return f"{title}\n\n{t['none_found']}\n\n"
    lines = []
    for h in items:
        label = (h.get('name_ar') if name == 'Arabic' else h.get('name_en')) or h['name']
        extra = [f"{h['distance_km']} {t['km']}"]
        if h.get('emergency') == 'yes':
            extra.append(t['er_yes'])
        if h.get('opening_hours'):
            extra.append(f"{t['hours']}: {_clean(h['opening_hours'])}")
        if h.get('phone'):
            extra.append(f"{t['phone']}: {_clean(h['phone'])}")
        lines.append(f"- **{_clean(label)}** — {' · '.join(extra)} — [{t['open_maps']}]({h['maps_url']})")
    return f"{title}\n" + '\n'.join(lines) + "\n\n"

# ============ Agent call + result formatting ============
def run_agent(request, location, facility, lang):
    t = UI[lang]
    name = t['lang_name']
    try:
        r, info, facilities = ask(request, name, location, facility)
    except Exception as e:
        return wrap(f"⚠️ {t['error']}\n\n`{e}`", name)
    urgent = info['urgent'] or r.urgent
    if urgent:
        dept = DEPT_NAMES['Emergency'][name]
    elif info['matched']:
        dept = DEPT_NAMES[info['department']][name]
    else:
        dept = r.department
    banner = f"> 🚨 **{t['urgent_banner']}**\n>\n> {t['urgent_call']}\n\n" if urgent else ''
    steps = '\n'.join(f'- {s}' for s in r.next_steps)
    body = (f"{banner}### {t['department']}: {dept}\n\n"
            f"**{t['reason']}:** {r.reason}\n\n"
            f"**{t['steps_title']}:**\n{steps}\n\n"
            f"{facilities_md(facilities, t, name)}"
            f"---\n*{t['disclaimer']}*")
    return wrap(body, name)

# ============ Journey handlers ============
def choose_language(lang):
    t = UI[lang]
    name = t['lang_name']
    return (
        lang,
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(value=wrap(t['goal'], name)),
        gr.update(label=t['input_label'], placeholder=t['placeholder'], value=''),
        gr.update(label=t['loc_label'], placeholder=t['loc_placeholder'], value=''),
        gr.update(label=t['fac_label'], value='hospital',
                  choices=[(t['fac_hospital'], 'hospital'), (t['fac_clinic'], 'clinic')]),
        gr.update(value=t['back']),
        gr.update(value=t['find']),
        gr.update(value=t['new']),
        gr.update(value=''),
        gr.update(value=stepper(2, t['steps'], name == 'Arabic')),
        gr.update(visible=(name == 'Arabic')),    # Arabic examples
        gr.update(visible=(name == 'English')),   # English examples
    )

def start_search(request, location, facility, lang):
    '''Step 1 (instant): switch to the result screen and show the loading animation.'''
    t = UI[lang]
    name = t['lang_name']
    text = (request or '').strip()
    if len(text) < 6:
        return (gr.update(visible=True), gr.update(visible=False), gr.update(value=''),
                gr.update(value=wrap(f"⚠️ {t['too_short']}", name)),
                gr.update(value=stepper(2, t['steps'], name == 'Arabic')), gr.update(visible=False))
    loading = wrap(f"<div class=\"loading\"><span class=\"dots\"><i></i><i></i><i></i></span> {t['analyzing']}</div>", name)
    return (gr.update(visible=False), gr.update(visible=True), gr.update(value=loading), gr.update(value=''),
            gr.update(value=stepper(3, t['steps'], name == 'Arabic')), gr.update(visible=False))

WATCHDOG_S = 120      # never leave the loading card on screen longer than this

def finish_search(request, location, facility, lang):
    '''Step 2 (slow): run the agent and replace the loading card with the result.
    Whatever happens (error, hang), the loading card is replaced by something readable.'''
    text = (request or '').strip()
    if len(text) < 6:
        return gr.update(), gr.update()
    t = UI[lang]
    name = t['lang_name']
    box = {}
    def work():
        try:
            box['md'] = run_agent(text, location or '', facility or 'hospital', lang)
        except BaseException as e:                       # anything run_agent did not catch
            box['err'] = e
            traceback.print_exc()                        # full traceback in the Colab cell output
    worker = threading.Thread(target=work, daemon=True)
    worker.start()
    worker.join(timeout=WATCHDOG_S)
    if worker.is_alive():
        md = wrap(f"⚠️ {t['error']}\n\n`timeout after {WATCHDOG_S} s`", name)
    elif 'err' in box:
        md = wrap(f"⚠️ {t['error']}\n\n`{type(box['err']).__name__}: {str(box['err'])[:200]}`", name)
    else:
        md = box['md']
    return gr.update(value=md), gr.update(visible=True)

def go_back():
    return gr.update(visible=True), gr.update(visible=False), gr.update(value=stepper(1, DEFAULT_STEPS))

def new_consultation(lang):
    t = UI[lang]
    return (gr.update(visible=True), gr.update(visible=False), gr.update(value=''),
            gr.update(value=''), gr.update(value=stepper(2, t['steps'], t['lang_name'] == 'Arabic')))

# ============ Build the interface ============
THEME = gr.themes.Soft(primary_hue='teal', secondary_hue='emerald')

# css/theme go in Blocks(...) in some Gradio versions and in launch(...) in others
css_in_blocks = 'css' in inspect.signature(gr.Blocks.__init__).parameters
blocks_kwargs = {'title': 'Hospital Guide Agent'}
launch_kwargs = {'share': True}
_launch_params = inspect.signature(gr.Blocks.launch).parameters
if 'show_error' in _launch_params:          # show errors in the page (the cell finishes right after launch)
    launch_kwargs['show_error'] = True
(blocks_kwargs if css_in_blocks else launch_kwargs).update(css=CSS, theme=THEME)

# fast navigation: skip the queue and hide the loading indicator
FAST = dict(queue=False, show_progress='hidden')

with gr.Blocks(**blocks_kwargs) as demo:
    lang_state = gr.State('English')
    gr.HTML(HERO)
    stepper_box = gr.HTML(stepper(1, DEFAULT_STEPS))

    # Screen 1: choose language
    with gr.Column(visible=True, elem_classes=['screen']) as screen1:
        gr.Markdown('### اختر لغتك · Choose your language')
        with gr.Row():
            btn_ar = gr.Button('العربية', variant='primary', elem_classes=['lang-btn'])
            btn_en = gr.Button('English', variant='primary', elem_classes=['lang-btn'])

    # Screen 2: describe the complaint (+ optional location)
    with gr.Column(visible=False, elem_classes=['screen']) as screen2:
        header = gr.Markdown()
        request_box = gr.Textbox(lines=4)
        location_box = gr.Textbox(lines=1, max_lines=1)
        facility_radio = gr.Radio(choices=[('hospital', 'hospital'), ('clinic', 'clinic')], value='hospital')
        hint = gr.Markdown()
        with gr.Row():
            back_btn = gr.Button()
            find_btn = gr.Button(variant='primary')

        # Examples: one block per language, only the chosen language is shown
        with gr.Column(visible=False) as ex_ar:
            gr.Examples(examples=[[x] for x in UI['العربية']['examples']],
                        inputs=[request_box], label=UI['العربية']['examples_label'])
        with gr.Column(visible=False) as ex_en:
            gr.Examples(examples=[[x] for x in UI['English']['examples']],
                        inputs=[request_box], label=UI['English']['examples_label'])

    # Screen 3: result
    with gr.Column(visible=False, elem_classes=['screen']) as screen3:
        output = gr.Markdown(elem_classes=['result'])
        new_btn = gr.Button(variant='primary')

    # Footer: contact links (shown on every screen)
    gr.HTML(FOOTER)

    choose_outputs = [lang_state, screen1, screen2, header, request_box, location_box, facility_radio,
                      back_btn, find_btn, new_btn, hint, stepper_box, ex_ar, ex_en]
    btn_ar.click(lambda: choose_language('العربية'), None, choose_outputs, **FAST)
    btn_en.click(lambda: choose_language('English'), None, choose_outputs, **FAST)
    search_inputs = [request_box, location_box, facility_radio, lang_state]
    find_btn.click(start_search, search_inputs, [screen2, screen3, output, hint, stepper_box, new_btn], **FAST
                   ).then(finish_search, search_inputs, [output, new_btn], show_progress='hidden')
    back_btn.click(go_back, None, [screen1, screen2, stepper_box], **FAST)
    new_btn.click(new_consultation, [lang_state], [screen2, screen3, request_box, hint, stepper_box], **FAST)

demo.launch(**launch_kwargs)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://155353ea995f2d66e6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7 · Test with THREE inputs
Try one normal, one detailed, and one tricky/empty input. Note what breaks.

## 🚀 Stretch (optional)
- Add a second tool.
- Add a safeguard step that flags unrealistic answers (Day 1 callback).
- Use `gr.Blocks` to add a dropdown or slider.

## Submit on Google Classroom
1. Your Colab link (sharing on).
2. Your Gradio public link or a screenshot.
3. Three sentences: what it does · which tool it uses · one thing that broke and how you fixed it.